# Test-Set Inference — Patent Draft Generator (Manus API)

Runs one task at a time using Manus API (`https://api.manus.ai/v2`).  

Manus is an async agent, not chat-completions. Flow:
1. `POST /v2/task.create` with `{"message": {"content": <prompt>}}` returns `task_id`.
2. Poll `GET /v2/task.listMessages?task_id=...` until a `status_update` has `agent_status == "stopped"`.
3. Concatenate `assistant_message.content` from every `assistant_message` event = prediction.

No temperature/top_p/max_tokens/model params

## 1. Install Dependencies

In [ ]:
!pip install -q requests tqdm python-dotenv

## 2. Manus API Key

Put `MANUS_API_KEY=...` in a `.env` file at the repo root (gitignored).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads MANUS_API_KEY from .env
MANUS_API_KEY = os.environ["MANUS_API_KEY"]

## 3. Configuration

Set `TASK` to `"task_1"` or `"task_2"`. `AGENT_PROFILE` is optional; `None` uses the default agent.

In [ ]:
# task
TASK = "task_2"  # task_1 | task_2

MANUS_API_BASE = "https://api.manus.ai/v2"
MODEL_TAG      = "manus"

# i/o paths, relative to repo root
TEST_SET_PATH = f"patent_datasets/{TASK}_test.json"
OUTPUT_DIR    = "patent_eval"
OUTPUT_PATH   = f"{OUTPUT_DIR}/predictions_{TASK}_{MODEL_TAG}.json"

# polling
POLL_INTERVAL   = 5      # sec between status checks
MAX_WAIT_SEC    = 3600   # give up on a task after this
REQUEST_TIMEOUT = 120    # per-request timeout, sec

print(f"Task   : {TASK}")
print(f"Output : {OUTPUT_PATH}")

## 4. Load Test Set

In [ ]:
import json

with open(TEST_SET_PATH, "r", encoding="utf-8") as f:
    test_items = json.load(f)

print(f"Test set : {TEST_SET_PATH}")
print(f"Samples  : {len(test_items)}")
print(f"Keys[0]  : {list(test_items[0].keys())}")

## 5. Manus API Client — create task → poll → collect output

In [ ]:
import time
import requests

H = {"x-manus-api-key": MANUS_API_KEY, "Content-Type": "application/json"}


def run_manus(prompt_text):
    # 1. create task
    body = {"message": {"content": prompt_text + "\n\n\nRESULT MUST BE SAVED TO output.json"},
            "agent_profile": "manus-1.6"}
    tid = requests.post(f"{MANUS_API_BASE}/task.create", headers=H,
                        json=body, timeout=REQUEST_TIMEOUT).json()["task_id"]

    # 2. poll the message list until the agent reports "stopped"
    t0 = time.time()
    while True:
        msgs = requests.get(f"{MANUS_API_BASE}/task.listMessages", headers=H,
                            params={"task_id": tid, "order": "asc", "limit": 200},
                            timeout=REQUEST_TIMEOUT).json()["messages"]
        status = next(((m.get("status_update") or {}).get("agent_status")
                       for m in reversed(msgs) if m["type"] == "status_update"), None)
        if status == "stopped":
            break
        if time.time() - t0 > MAX_WAIT_SEC:
            raise TimeoutError(f"task {tid} not done after {MAX_WAIT_SEC}s (last status={status})")
        time.sleep(POLL_INTERVAL)

    # 3. find the output.txt attachment the agent generated, then download it
    url = None
    for m in msgs:
        if m["type"] == "assistant_message":
            for att in m["assistant_message"].get("attachments") or []:
                if (att.get("filename") or "").endswith(".json"):
                    url = att["url"]  
    if url is None:
        raise RuntimeError(f"task {tid}: no output.txt attachment found")

    text = requests.get(url, timeout=REQUEST_TIMEOUT).text
    return text.strip(), tid


requests.get(f"{MANUS_API_BASE}/task.list", headers=H,
             params={"limit": 1}, timeout=REQUEST_TIMEOUT).raise_for_status()
print("Manus API auth OK.")

## 6. Build Prompt

Manus messages have a single `content` field (no system role), so instruction and input are concatenated.

In [ ]:
def build_prompt(entry: dict) -> str:
    return entry["instruction"] + "\n\n" + entry["input"]

_p = build_prompt(test_items[0])
print(_p[:1200], "\n...")

## 7. Generate Predictions for the Whole Test Set

In [ ]:
import os
import time
from tqdm.auto import tqdm

os.makedirs(OUTPUT_DIR, exist_ok=True)

# resume: skip indices already in output file
predictions = []
done_indices = set()
if os.path.exists(OUTPUT_PATH):
    try:
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            existing = json.load(f)
        predictions = existing.get("predictions", [])
        done_indices = {p["index"] for p in predictions}
        print(f"Resuming — {len(done_indices)} predictions already on disk.")
    except Exception as e:
        print(f"Could not resume ({e}); starting fresh.")


def save_checkpoint():
    predictions.sort(key=lambda p: p["index"])
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "_meta": {
                "task": TASK, "base_model": "manus", "adapter": None,
                "api": "manus", "api_base": MANUS_API_BASE,
                "decoding": "agent",
                "test_set_path": TEST_SET_PATH,
                "test_set_size": len(test_items),
            },
            "predictions": predictions,
        }, f, ensure_ascii=False, indent=2)


pending = [(i, e) for i, e in enumerate(test_items) if i not in done_indices]

started = time.time()
for idx, entry in tqdm(pending, desc=f"Generating {TASK}/{MODEL_TAG} (Manus API)"):
    prompt_text = build_prompt(entry)

    _gen_t0 = time.perf_counter()
    pred, task_id = run_manus(prompt_text)
    gen_time = time.perf_counter() - _gen_t0

    predictions.append({
        "index":         idx,
        "instruction":   entry["instruction"],
        "input":         entry["input"],
        "reference":     entry["output"],
        "prediction":    pred,
        "input_tokens":  0,            
        "output_tokens": 0,
        "gen_time_sec":  round(gen_time, 3),
        "manus_task_id": task_id,
    })

    save_checkpoint()  # per sample

elapsed = time.time() - started
print(f"\nGenerated {len(predictions)} predictions in {elapsed/60:.1f} min.")

## 8. Final Save

In [ ]:
save_checkpoint()
print(f"Saved {len(predictions)} predictions to {OUTPUT_PATH}")